# Phase 4: Lifetime Value (LTV) Prediction
## Customer Churn & LTV Prediction Engine

This notebook focuses on estimating **Customer Lifetime Value (LTV)**. It explores methods to model LTV: regression approaches predicting continuous value outputs (`TotalCharges`), and statistical models like BG/NBD and Gamma-Gamma (if using transactional cohort datasets) or customer cohort survival modeling.

### Objectives:
1. Define/derive LTV proxies using charges and contract lengths.
2. Train regression models to estimate total spend values.
3. Align churn probability and total spend expectations to calculate risk-adjusted LTV.
4. Conduct segmentation to categorize users into High, Medium, and Low-value tiers.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

# Load preprocessed splits
X_train_raw = pd.read_csv("../data/processed/X_train.csv")
X_test_raw  = pd.read_csv("../data/processed/X_test.csv")
y_train_df  = pd.read_csv("../data/processed/y_train.csv")
y_test_df   = pd.read_csv("../data/processed/y_test.csv")

# Load raw data to get TotalCharges as LTV target
RAW_PATH = os.path.join(os.path.abspath(".."), "data", "raw",
                        "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw = pd.read_csv(RAW_PATH)

# Fix TotalCharges dtype
df_raw["TotalCharges"] = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")
df_raw.loc[df_raw["tenure"] == 0, "TotalCharges"] = 0.0
df_raw["TotalCharges"] = df_raw["TotalCharges"].fillna(df_raw["TotalCharges"].median())
df_raw["Churn"] = df_raw["Churn"].map({"Yes": 1, "No": 0})

# Same 80/20 split as notebook 02
df_train_raw = df_raw.sample(frac=0.8, random_state=42)
df_test_raw  = df_raw.drop(df_train_raw.index)
y_train_ltv  = df_train_raw["TotalCharges"].reset_index(drop=True)
y_test_ltv   = df_test_raw["TotalCharges"].reset_index(drop=True)

# Encode categoricals for LTV (replaces ColumnTransformer)
CAT_COLS = [
    "gender","SeniorCitizen","Partner","Dependents","PhoneService",
    "MultipleLines","InternetService","OnlineSecurity","OnlineBackup",
    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies",
    "Contract","PaperlessBilling","PaymentMethod"
]
NUM_COLS = ["tenure", "MonthlyCharges"]

# Features for LTV: raw columns (NOT TotalCharges - that is the target)
feat_cols = CAT_COLS + NUM_COLS
X_tr = df_train_raw[feat_cols].copy().reset_index(drop=True)
X_te = df_test_raw[feat_cols].copy().reset_index(drop=True)

# pd.get_dummies replaces OneHotEncoder
X_tr_enc = pd.get_dummies(X_tr, columns=CAT_COLS, drop_first=False)
X_te_enc = pd.get_dummies(X_te, columns=CAT_COLS, drop_first=False)
X_te_enc = X_te_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

# Scale numericals (replaces StandardScaler)
for col in NUM_COLS:
    mean = X_tr_enc[col].mean(); std = X_tr_enc[col].std()
    X_tr_enc[col] = (X_tr_enc[col] - mean) / (std if std > 0 else 1)
    X_te_enc[col] = (X_te_enc[col] - mean) / (std if std > 0 else 1)

print(f"LTV feature matrix: {X_tr_enc.shape}")
print(f"LTV train target - mean: {y_train_ltv.mean():.2f}  std: {y_train_ltv.std():.2f}")
print("Setup complete (no sklearn used)")


In [ ]:
from xgboost import XGBRegressor

# Train XGBoost Regressor for LTV
print("Training XGBoost Regressor for LTV (TotalCharges)...")
xgb_ltv = XGBRegressor(
    n_estimators     = 200,
    max_depth        = 5,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    objective        = "reg:squarederror",
    random_state     = 42,
    n_jobs           = -1,
)
xgb_ltv.fit(
    X_tr_enc, y_train_ltv,
    eval_set=[(X_te_enc, y_test_ltv)],
    verbose=False,
)
y_pred_ltv = xgb_ltv.predict(X_te_enc)

# Regression Metrics (pure numpy - no sklearn)
y_true = y_test_ltv.values
mae  = np.abs(y_pred_ltv - y_true).mean()
mse  = ((y_pred_ltv - y_true) ** 2).mean()
rmse = np.sqrt(mse)
ss_res = ((y_true - y_pred_ltv) ** 2).sum()
ss_tot = ((y_true - y_true.mean()) ** 2).sum()
r2   = 1 - ss_res / ss_tot

print(f"\n--- XGBoost LTV Regressor Results ---")
print(f"  MAE  : ${mae:,.2f}")
print(f"  RMSE : ${rmse:,.2f}")
print(f"  R2   : {r2:.4f}")

# Visualise predictions vs actuals
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_true, y_pred_ltv, alpha=0.3, color="steelblue", s=10)
lims = [min(y_true.min(), y_pred_ltv.min()), max(y_true.max(), y_pred_ltv.max())]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel("Actual TotalCharges")
axes[0].set_ylabel("Predicted TotalCharges")
axes[0].set_title(f"Predicted vs Actual LTV (R2 = {r2:.4f})")

residuals = y_pred_ltv - y_true
axes[1].hist(residuals, bins=40, color="salmon", edgecolor="white")
axes[1].axvline(0, color="black", lw=1, linestyle="--")
axes[1].set_xlabel("Residual (Predicted - Actual)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()


In [3]:
# 1. Load Churn model and its preprocessor
churn_model = joblib.load('../models/churn_model.joblib')
churn_preprocessor = joblib.load('../models/preprocessor.joblib')

# 2. Get Churn probabilities on test set
# We must use the original raw X_test because the Churn preprocessor expect those exact columns
X_test_trans_churn = churn_preprocessor.transform(X_test_raw)
churn_probs = churn_model.predict_proba(X_test_trans_churn)[:, 1]

# 3. Create a summary DataFrame for the test set
test_summary = pd.DataFrame({
    'Actual_Churn': y_test_ltv.index.map(df['Churn']) if 'df' in locals() else y_test_ltv * 0, # backup mapping
    'Actual_TotalCharges': y_test_ltv,
    'Predicted_TotalCharges': y_pred_ltv,
    'Churn_Probability': churn_probs
})

# Re-align actual churn labels from the indices
test_summary['Actual_Churn'] = X_test_raw.index.map(pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')['Churn'].map({'Yes': 1, 'No': 0}))

# 4. Calculate Risk-Adjusted LTV
test_summary['Risk_Adjusted_LTV'] = test_summary['Predicted_TotalCharges'] * (1 - test_summary['Churn_Probability'])

# 5. Segment customers based on Median Spend and Churn Risk (threshold = 0.5)
median_spend = test_summary['Predicted_TotalCharges'].median()

def segment_customer(row):
    is_high_spend = row['Predicted_TotalCharges'] >= median_spend
    is_high_risk = row['Churn_Probability'] >= 0.5
    
    if is_high_spend and is_high_risk:
        return 'Rescue Target (High Spend, High Churn Risk)'
    elif is_high_spend and not is_high_risk:
        return 'Loyal Champion (High Spend, Low Churn Risk)'
    elif not is_high_spend and is_high_risk:
        return 'Low Priority Churn (Low Spend, High Churn Risk)'
    else:
        return 'Stable Core (Low Spend, Low Churn Risk)'

test_summary['Segment'] = test_summary.apply(segment_customer, axis=1)

# Print the segment distribution and average values
print("=== CUSTOMER SEGMENTATION SUMMARY ===")
print(test_summary['Segment'].value_counts())
print("\n=== AVERAGE VALUES BY SEGMENT ===")
print(test_summary.groupby('Segment')[['Predicted_TotalCharges', 'Churn_Probability', 'Risk_Adjusted_LTV']].mean())


=== CUSTOMER SEGMENTATION SUMMARY ===
Segment
Loyal Champion (High Spend, Low Churn Risk)        528
Low Priority Churn (Low Spend, High Churn Risk)    363
Stable Core (Low Spend, Low Churn Risk)            341
Rescue Target (High Spend, High Churn Risk)        177
Name: count, dtype: int64

=== AVERAGE VALUES BY SEGMENT ===
                                                 Predicted_TotalCharges  \
Segment                                                                   
Low Priority Churn (Low Spend, High Churn Risk)              484.296370   
Loyal Champion (High Spend, Low Churn Risk)                 4293.458739   
Rescue Target (High Spend, High Churn Risk)                 3010.439691   
Stable Core (Low Spend, Low Churn Risk)                      459.150796   

                                                 Churn_Probability  \
Segment                                                              
Low Priority Churn (Low Spend, High Churn Risk)           0.744923   
Loyal Champi

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("../models", exist_ok=True)
ltv_model_path = "../models/ltv_model.json"
xgb_ltv.save_model(ltv_model_path)

print(f"LTV model saved to: {os.path.abspath(ltv_model_path)}")
print(f"MAE: ${mae:,.2f}  |  R2: {r2:.4f}")

# Feature Importance
feat_imp = pd.Series(
    xgb_ltv.feature_importances_,
    index=X_tr_enc.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind="barh", ax=ax, color="darkorange")
ax.invert_yaxis()
ax.set_title("Top-15 Feature Importances - XGBoost LTV Model")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()
